In [2]:
import duckdb
import pandas as pd
from nyc_taxi.config.settings import DATABASE_PATH, NYC_TAXI_DIR
import pugsql
from prefect import get_run_logger, task

con = duckdb.connect(DATABASE_PATH)
queries = pugsql.module(str(NYC_TAXI_DIR / 'queries'))

yellow_preview = con.execute("""
    SELECT *
    FROM silver.yellow_trips
    LIMIT 5
""").df()

green_preview = con.execute("""
    SELECT *
    FROM silver.green_trips
    LIMIT 5
""").df()

yellow_preview

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_location_id,dropoff_location_id,fare_amount,tip_amount,total_amount,trip_duration_min,location_id,pickup_borough,pickup_zone,pickup_hour,day_name,tip_percentage,temperature_2m_max,precipitation_sum,weathercode,is_raining
0,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,229,237,10.0,3.00,18.00,8.35,229,Manhattan,Sutton Place/Turtle Bay North,0,Wednesday,30.00,10.9,4.5,63,True
1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,236,237,5.1,2.02,12.12,2.55,236,Manhattan,Upper East Side North,0,Wednesday,39.61,10.9,4.5,63,True
2,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,141,141,5.1,2.00,12.10,1.95,141,Manhattan,Lenox Hill West,0,Wednesday,39.22,10.9,4.5,63,True
3,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,244,244,7.2,0.00,9.70,5.57,244,Manhattan,Washington Heights South,0,Wednesday,0.00,10.9,4.5,63,True
4,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,244,116,5.8,0.00,8.30,3.53,244,Manhattan,Washington Heights South,0,Wednesday,0.00,10.9,4.5,63,True


In [ ]:
yellow_cols = con.execute("""
    DESCRIBE SELECT *
    FROM silver.yellow_trips
""").df()

green_cols = con.execute("""
    DESCRIBE SELECT *
    FROM silver.green_trips
""").df()

print("YELLOW COLUMNS")
display(yellow_cols)

print("GREEN COLUMNS")
display(green_cols)

YELLOW COLUMNS


,column_name,column_type,null,key,default,extra
0,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
1,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
2,passenger_count,DOUBLE,YES,None,None,None
3,trip_distance,DOUBLE,YES,None,None,None
4,pickup_location_id,INTEGER,YES,None,None,None
5,dropoff_location_id,INTEGER,YES,None,None,None
6,fare_amount,DOUBLE,YES,None,None,None
7,tip_amount,DOUBLE,YES,None,None,None
8,total_amount,DOUBLE,YES,None,None,None
9,trip_duration_min,DOUBLE,YES,None,None,None


GREEN COLUMNS


,column_name,column_type,null,key,default,extra
0,lpep_pickup_datetime,TIMESTAMP,YES,None,None,None
1,lpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
2,pickup_location_id,INTEGER,YES,None,None,None
3,dropoff_location_id,INTEGER,YES,None,None,None
4,passenger_count,DOUBLE,YES,None,None,None
5,trip_distance,DOUBLE,YES,None,None,None
6,fare_amount,DOUBLE,YES,None,None,None
7,tip_amount,DOUBLE,YES,None,None,None
8,total_amount,DOUBLE,YES,None,None,None
9,trip_duration_min,DOUBLE,YES,None,None,None


In [3]:
zone_lookup = con.execute("""
    SELECT *
    FROM read_csv_auto('../data/static/taxi_zone_lookup.csv')
    LIMIT 10
""").df()

zone_lookup

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
5,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
6,7,Queens,Astoria,Boro Zone
7,8,Queens,Astoria Park,Boro Zone
8,9,Queens,Auburndale,Boro Zone
9,10,Queens,Baisley Park,Boro Zone


In [3]:
con.close()

In [8]:
@task(name="Fetch Gold Profitability Data", retries=3)
def fetch_gold_profitability():
    logger = get_run_logger()

    import duckdb

    query_path = NYC_TAXI_DIR / "queries" / "silver_to_gold_profitability.sql"
    sql = query_path.read_text()

    try:
        with duckdb.connect(str(DATABASE_PATH), read_only=True) as con:
            df_result = con.execute(sql).df()

        logger.info("Data profitabilitas berhasil ditarik ke DataFrame via DuckDB native!")

    except Exception as e:
        logger.error(f"Gagal menarik data: {e}")
        raise

    return df_result

In [10]:
# os.makedirs('../data/gold', exist_ok=True)

df_zone_profit = fetch_gold_profitability()

22:40:18.797 | INFO    | Task run 'Fetch Gold Profitability Data' - Data profitabilitas berhasil ditarik ke DataFrame via DuckDB native!

C:\Users\USER\AppData\Roaming\uv\python\cpython-3.13.5-windows-x86_64-none\Lib\logging\__init__.py:1957: UserWarning: Logger 'prefect.task_runs' attempted to send logs to the API without a flow run id. The API log handler can only send logs within flow run contexts unless the flow run id is manually provided. Set PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW=ignore to suppress this warning.
  self.logger.log(level, msg, *args, **kwargs)


22:40:18.800 | INFO    | Task run 'Fetch Gold Profitability Data' - Finished in state Completed()

In [11]:
df_zone_profit[df_zone_profit["service_type"] == "Yellow"].head(10)

,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
0,Yellow,132,JFK Airport,Queens,534830,43685154.78,62.97,9.44
1,Yellow,138,LaGuardia Airport,Queens,373217,25320133.68,42.92,9.24
2,Yellow,161,Midtown Center,Manhattan,651742,16193310.03,15.74,3.14
3,Yellow,237,Upper East Side South,Manhattan,640757,13151721.73,12.70,2.64
4,Yellow,230,Times Sq/Theatre District,Manhattan,472054,13000917.94,18.00,3.32
5,Yellow,236,Upper East Side North,Manhattan,582236,12148732.10,13.27,2.64
6,Yellow,186,Penn Station/Madison Sq West,Manhattan,463469,11395352.87,15.92,3.09
7,Yellow,162,Midtown East,Manhattan,456040,11069113.65,15.24,3.13
8,Yellow,163,Midtown North,Manhattan,377477,9396883.79,15.86,3.14
9,Yellow,142,Lincoln Square East,Manhattan,419672,9266634.58,14.04,2.79


In [12]:
df_zone_profit[df_zone_profit["service_type"] == "Green"].head(10)

,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
51,Green,74,East Harlem North,Manhattan,45112,945246.29,14.86,2.72
60,Green,75,East Harlem South,Manhattan,30114,643772.67,14.54,2.57
95,Green,43,Central Park,Manhattan,10094,231434.86,14.95,3.06
98,Green,166,Morningside Heights,Manhattan,9642,214949.76,15.68,2.89
107,Green,95,Forest Hills,Queens,8677,183496.78,16.43,1.93
112,Green,82,Elmhurst,Queens,6669,160200.16,18.07,2.02
113,Green,244,Washington Heights South,Manhattan,4539,159335.82,26.63,4.17
116,Green,97,Fort Greene,Brooklyn,6710,152988.89,17.21,2.90
117,Green,130,Jamaica,Queens,5316,149149.70,22.42,2.74
119,Green,41,Central Harlem,Manhattan,7570,147984.91,14.50,1.93


In [13]:
print("TOP 10 YELLOW BY TOTAL REVENUE")
display(
    df_zone_profit[df_zone_profit["service_type"] == "Yellow"]
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

print("TOP 10 GREEN BY TOTAL REVENUE")
display(
    df_zone_profit[df_zone_profit["service_type"] == "Green"]
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

TOP 10 YELLOW BY TOTAL REVENUE


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
0,Yellow,132,JFK Airport,Queens,534830,43685154.78,62.97,9.44
1,Yellow,138,LaGuardia Airport,Queens,373217,25320133.68,42.92,9.24
2,Yellow,161,Midtown Center,Manhattan,651742,16193310.03,15.74,3.14
3,Yellow,237,Upper East Side South,Manhattan,640757,13151721.73,12.70,2.64
4,Yellow,230,Times Sq/Theatre District,Manhattan,472054,13000917.94,18.00,3.32
5,Yellow,236,Upper East Side North,Manhattan,582236,12148732.10,13.27,2.64
6,Yellow,186,Penn Station/Madison Sq West,Manhattan,463469,11395352.87,15.92,3.09
7,Yellow,162,Midtown East,Manhattan,456040,11069113.65,15.24,3.13
8,Yellow,163,Midtown North,Manhattan,377477,9396883.79,15.86,3.14
9,Yellow,142,Lincoln Square East,Manhattan,419672,9266634.58,14.04,2.79


TOP 10 GREEN BY TOTAL REVENUE


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
51,Green,74,East Harlem North,Manhattan,45112,945246.29,14.86,2.72
60,Green,75,East Harlem South,Manhattan,30114,643772.67,14.54,2.57
95,Green,43,Central Park,Manhattan,10094,231434.86,14.95,3.06
98,Green,166,Morningside Heights,Manhattan,9642,214949.76,15.68,2.89
107,Green,95,Forest Hills,Queens,8677,183496.78,16.43,1.93
112,Green,82,Elmhurst,Queens,6669,160200.16,18.07,2.02
113,Green,244,Washington Heights South,Manhattan,4539,159335.82,26.63,4.17
116,Green,97,Fort Greene,Brooklyn,6710,152988.89,17.21,2.90
117,Green,130,Jamaica,Queens,5316,149149.70,22.42,2.74
119,Green,41,Central Harlem,Manhattan,7570,147984.91,14.50,1.93


In [14]:
print("TOP 10 YELLOW BY AVG FARE")
display(
    df_zone_profit[df_zone_profit["service_type"] == "Yellow"]
    .sort_values("avg_fare", ascending=False)
    .head(10)
)

print("TOP 10 GREEN BY AVG FARE")
display(
    df_zone_profit[df_zone_profit["service_type"] == "Green"]
    .sort_values("avg_fare", ascending=False)
    .head(10)
)

TOP 10 YELLOW BY AVG FARE


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
425,Yellow,44,Charleston/Tottenville,Staten Island,3,336.22,86.13,0.00
260,Yellow,1,Newark Airport,EWR,95,10468.28,85.58,12.68
422,Yellow,204,Rossville/Woodrow,Staten Island,4,350.48,73.50,0.00
128,Yellow,265,Outside of NYC,Unknown,1479,126166.79,71.05,7.81
470,Yellow,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,1,90.69,70.00,9.00
418,Yellow,84,Eltingville/Annadale/Prince's Bay,Staten Island,5,381.43,67.46,1.86
0,Yellow,132,JFK Airport,Queens,534830,43685154.78,62.97,9.44
419,Yellow,5,Arden Heights,Staten Island,5,360.27,57.89,0.00
114,Yellow,93,Flushing Meadows-Corona Park,Queens,2178,154294.44,57.48,7.06
388,Yellow,187,Port Richmond,Staten Island,9,581.71,54.15,1.62


TOP 10 GREEN BY AVG FARE


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
459,Green,245,West Brighton,Staten Island,1,144.98,99.60,25.00
473,Green,46,City Island,Bronx,1,82.14,70.20,0.00
373,Green,86,Far Rockaway,Queens,12,805.84,63.83,0.17
456,Green,201,Rockaway Park,Queens,2,158.52,63.11,4.61
474,Green,6,Arrochar/Fort Wadsworth,Staten Island,1,73.33,61.39,0.00
390,Green,64,Douglaston,Queens,8,570.91,60.57,1.92
299,Green,265,Outside of NYC,Unknown,47,3079.53,57.50,5.04
302,Green,219,Springfield Gardens South,Queens,41,2924.50,57.49,5.24
393,Green,154,Marine Park/Floyd Bennett Field,Brooklyn,9,525.92,56.05,0.00
375,Green,117,Hammels/Arverne,Queens,13,796.11,55.56,0.54


In [15]:
print("TOP 10 YELLOW BY AVG TIP")
display(
    df_zone_profit[df_zone_profit["service_type"] == "Yellow"]
    .sort_values("avg_tip", ascending=False)
    .head(10)
)

print("TOP 10 GREEN BY AVG TIP")
display(
    df_zone_profit[df_zone_profit["service_type"] == "Green"]
    .sort_values("avg_tip", ascending=False)
    .head(10)
)

TOP 10 YELLOW BY AVG TIP


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
260,Yellow,1,Newark Airport,EWR,95,10468.28,85.58,12.68
0,Yellow,132,JFK Airport,Queens,534830,43685154.78,62.97,9.44
1,Yellow,138,LaGuardia Airport,Queens,373217,25320133.68,42.92,9.24
470,Yellow,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,1,90.69,70.00,9.00
343,Yellow,2,Jamaica Bay,Queens,19,1287.98,52.78,8.65
40,Yellow,70,East Elmhurst,Queens,40040,2596073.66,42.07,8.52
410,Yellow,199,Rikers Island,Bronx,7,412.84,39.10,8.42
128,Yellow,265,Outside of NYC,Unknown,1479,126166.79,71.05,7.81
248,Yellow,207,Saint Michaels Cemetery/Woodside,Queens,293,17938.23,48.49,7.11
114,Yellow,93,Flushing Meadows-Corona Park,Queens,2178,154294.44,57.48,7.06


TOP 10 GREEN BY AVG TIP


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
459,Green,245,West Brighton,Staten Island,1,144.98,99.60,25.00
293,Green,89,Flatbush/Ditmas Park,Brooklyn,63,3673.54,41.73,7.03
414,Green,26,Borough Park,Brooklyn,7,401.90,32.29,6.85
401,Green,194,Randalls Island,Manhattan,10,477.89,30.71,6.51
483,Green,224,Stuy Town/Peter Cooper Village,Manhattan,1,38.80,27.33,6.47
265,Green,146,Long Island City/Queens Plaza,Queens,269,8910.89,22.56,6.38
239,Green,93,Flushing Meadows-Corona Park,Queens,421,24303.47,47.66,5.72
235,Green,157,Maspeth,Queens,529,25447.08,39.16,5.42
302,Green,219,Springfield Gardens South,Queens,41,2924.50,57.49,5.24
290,Green,34,Brooklyn Navy Yard,Brooklyn,73,3835.52,44.32,5.23


In [16]:
import json
import pandas as pd
from nyc_taxi.config.settings import STATIC_DIR

with open(STATIC_DIR / "NYC Taxi Zones.geojson", "r", encoding="utf-8") as f:
    geo = json.load(f)

print("Jumlah feature:", len(geo["features"]))
print("\nContoh properties feature pertama:")
print(geo["features"][0]["properties"])

Jumlah feature: 263

Contoh properties feature pertama:
{'shape_area': '0.0007823067885', 'objectid': '1', 'shape_leng': '0.116357453189', 'location_id': '1', 'zone': 'Newark Airport', 'borough': 'EWR'}


In [17]:
lookup = pd.read_csv(STATIC_DIR / "taxi_zone_lookup.csv")

print("Kolom lookup:")
print(lookup.columns.tolist())

print("\n5 baris pertama lookup:")
display(lookup.head())

Kolom lookup:
['LocationID', 'Borough', 'Zone', 'service_zone']

5 baris pertama lookup:


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [18]:
# zone_profit = pd.read_csv("../data/gold/zone_profitability.csv")
# df_zone_profit = fetch_gold_profitability()

print("Kolom zone_profitability:")
print(df_zone_profit.columns.tolist())

display(df_zone_profit.head())

Kolom zone_profitability:
['service_type', 'pickup_location_id', 'zone', 'borough', 'trip_count', 'total_revenue', 'avg_fare', 'avg_tip']


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
0,Yellow,132,JFK Airport,Queens,534830,43685154.78,62.97,9.44
1,Yellow,138,LaGuardia Airport,Queens,373217,25320133.68,42.92,9.24
2,Yellow,161,Midtown Center,Manhattan,651742,16193310.03,15.74,3.14
3,Yellow,237,Upper East Side South,Manhattan,640757,13151721.73,12.70,2.64
4,Yellow,230,Times Sq/Theatre District,Manhattan,472054,13000917.94,18.00,3.32


In [19]:
# import json
# import copy

# lookup = pd.read_csv("../data/static/taxi_zone_lookup.csv")
# zone_profit = pd.read_csv("../data/gold/zone_profitability.csv")

# Kalau belum ada pickup_location_id, ambil dari lookup pakai zone + borough
if "pickup_location_id" not in df_zone_profit.columns:
    df_zone_profit = df_zone_profit.merge(
        lookup[["LocationID", "Borough", "Zone"]],
        left_on=["borough", "zone"],
        right_on=["Borough", "Zone"],
        how="left"
    )
    df_zone_profit = df_zone_profit.rename(columns={"LocationID": "pickup_location_id"})

print("Kolom akhir df_zone_profit:")
print(df_zone_profit.columns.tolist())

print("\nJumlah baris tanpa pickup_location_id:")
print(df_zone_profit["pickup_location_id"].isna().sum())

display(df_zone_profit.head())

Kolom akhir df_zone_profit:
['service_type', 'pickup_location_id', 'zone', 'borough', 'trip_count', 'total_revenue', 'avg_fare', 'avg_tip']

Jumlah baris tanpa pickup_location_id:
0


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
0,Yellow,132,JFK Airport,Queens,534830,43685154.78,62.97,9.44
1,Yellow,138,LaGuardia Airport,Queens,373217,25320133.68,42.92,9.24
2,Yellow,161,Midtown Center,Manhattan,651742,16193310.03,15.74,3.14
3,Yellow,237,Upper East Side South,Manhattan,640757,13151721.73,12.70,2.64
4,Yellow,230,Times Sq/Theatre District,Manhattan,472054,13000917.94,18.00,3.32


In [20]:
# zone_profit.to_csv("../data/gold/zone_profitability.csv", index=False)
# print("zone_profitability.csv berhasil di-update")

from nyc_taxi.utils.db_utils import save_to_db

save_to_db(df_zone_profit, 'zone_profitability', 'gold')

In [20]:
con.close()